# ЛР3. Сервер статистики SQLite

Требуется сохранять время вызова, тип операции и её длительность в `log.db`.
Эта тетрадь заменяет CSV-сервис из ЛР2 и использует тот же порт 8018.
Старый `xmlrpc_stats_server.ipynb` одновременно запускать не нужно.
Запустите ячейку первой. Перезапуск сервиса не удаляет записи.
Перед повторным запуском перезапустите ядро, чтобы освободить порт.


In [1]:
from pathlib import Path
from xmlrpc.server import SimpleXMLRPCRequestHandler, SimpleXMLRPCServer
from datetime import datetime
import math
import sqlite3


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)


database_file = Path.cwd() / 'log.db'


# Таблица создаётся только при первом запуске; старые события сохраняются.
def create_database():
    with sqlite3.connect(database_file) as connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS events (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT NOT NULL,
                event_type TEXT NOT NULL,
                duration REAL NOT NULL,
                details TEXT NOT NULL
            )
        """)


# Время вызова передаёт прокси, а длительность хранится в секундах.
def add_event(event_type, timestamp, duration, details=''):
    datetime.strptime(timestamp, '%Y-%m-%d %H:%M:%S')
    duration = float(duration)
    if not math.isfinite(duration) or duration < 0:
        raise ValueError('Длительность должна быть конечным неотрицательным числом')
    with sqlite3.connect(database_file) as connection:
        connection.execute(
            'INSERT INTO events (timestamp, event_type, duration, details) VALUES (?, ?, ?, ?)',
            (timestamp, str(event_type), duration, str(details))
        )
    return True


# Пустые значения отключают фильтр. Все границы включаются в выборку.
def get_events(event_type='', start_time='', end_time='', min_duration=None, max_duration=None):
    conditions = []
    parameters = []
    if event_type:
        conditions.append('event_type = ?')
        parameters.append(event_type)
    for column_condition, time_value in [('timestamp >= ?', start_time), ('timestamp <= ?', end_time)]:
        if time_value:
            datetime.strptime(time_value, '%Y-%m-%d %H:%M:%S')
            conditions.append(column_condition)
            parameters.append(time_value)
    if start_time and end_time and start_time > end_time:
        raise ValueError('Начальное время не должно быть позже конечного')
    for column_condition, duration_value in [('duration >= ?', min_duration), ('duration <= ?', max_duration)]:
        if duration_value is not None:
            duration_value = float(duration_value)
            if not math.isfinite(duration_value) or duration_value < 0:
                raise ValueError('Граница длительности должна быть конечным неотрицательным числом')
            conditions.append(column_condition)
            parameters.append(duration_value)
    if min_duration is not None and max_duration is not None and float(min_duration) > float(max_duration):
        raise ValueError('Минимальная длительность не должна превышать максимальную')
    query = 'SELECT id, timestamp, event_type, duration, details FROM events'
    if conditions:
        query += ' WHERE ' + ' AND '.join(conditions)
    query += ' ORDER BY timestamp, id'
    with sqlite3.connect(database_file) as connection:
        connection.row_factory = sqlite3.Row
        return [dict(event_record) for event_record in connection.execute(query, parameters)]


create_database()
statistics_server = SimpleXMLRPCServer(
    ('127.0.0.1', 8018), requestHandler=RequestHandler, allow_none=True, logRequests=True
)
statistics_server.register_introspection_functions()
statistics_server.register_function(add_event, 'add_event')
statistics_server.register_function(get_events, 'get_events')

print('Сервер статистики SQLite слушает порт 8018', flush=True)
print('База данных:', database_file, flush=True)
try:
    statistics_server.serve_forever()
except KeyboardInterrupt:
    print("Сервер остановлен")
finally:
    statistics_server.server_close()


Сервер статистики SQLite слушает порт 8018
База данных: C:\Users\Kromobot\JupyterNotebook\РСОИ\ЛР\LR3\log.db
Сервер статистики SQLite слушает порт 8018
База данных: C:\Users\Kromobot\JupyterNotebook\РСОИ\ЛР\LR3\log.db
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:57] "POST /RPC2 HTTP/1.1" 200 -
Сервер статистики SQLite слушает порт 8018
База данных: C:\Users\Kromobot\JupyterNotebook\РСОИ\ЛР\LR3\log.db
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2